# L39 · 从零手写 FFT — Cooley-Tukey 递归实现，与 numpy.fft 误差 < 1e-10

**目标**：手写 Cooley-Tukey 递归 FFT，让 `tests/audio/test_transforms.py` 全绿。

🔗 **Aurora 连接**：本节实现镜像 `src/aurora/audio/transforms.py` 的 `fft()` 函数。

← **上一课**　[L38 · FFT 蝶形分治](L38_fft_butterfly.ipynb)

> 上节课学习了 **FFT 蝶形分治**：偶奇拆分、蝶形运算 E[k]+W^k·O[k]，O(N²)→O(N log N)。  
> 本课将探讨 **从零手写 FFT**。

FFT 的本质是**分治（divide and conquer）**：把 N 点 DFT 拆成两个 N/2 点 DFT，再用蝶形公式合并，把 O(N²) 变成 O(N log N)。每层递归只做旋转因子乘法和加减，整个计算树有 log₂N 层，每层 N 次运算。从零实现一遍，你对 STFT、mel 滤波器的数值行为就有了直觉基础。

In [ ]:
import numpy as np
import time
from aurora.audio.transforms import fft as aurora_fft

## 定位图：N=8 递归分治树

先看树的结构再动手写递归。向下是**拆分**（偶/奇），向上是**蝶形合并**；3 层对应 `log₂8=3` 次递归，每层 N/2=4 次蝶形乘法，总计 12 次——朴素 DFT 要 64 次。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(10, 4.2))

COLORS = {8: '#3498db', 4: '#e74c3c', 2: '#2ecc71', 1: '#95a5a6'}
LABELS = {8: 'my_fft(8)', 4: 'my_fft(4)', 2: 'my_fft(2)', 1: 'N=1\n返回原值'}
ys = [3.5, 2.4, 1.3, 0.2]   # y-centers per level
bh = 0.65                     # block height

def box(ax, cx, cy, w, n):
    ax.add_patch(plt.Rectangle((cx - w/2, cy - bh/2), w, bh,
                                fc=COLORS[n], ec='white', lw=2, alpha=0.88, zorder=3))
    ax.text(cx, cy, LABELS[n], ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white', zorder=4)

# Boxes at each level
box(ax, 5.0, ys[0], 9.0, 8)
for cx in [2.5, 7.5]:
    box(ax, cx, ys[1], 4.1, 4)
for cx in [1.25, 3.75, 6.25, 8.75]:
    box(ax, cx, ys[2], 1.75, 2)
for cx in np.linspace(0.6, 9.4, 8):
    box(ax, cx, ys[3], 0.68, 1)

# Arrows (split direction, going down)
kw = dict(arrowstyle='->', color='#7f8c8d', lw=1.2)
for cx in [2.5, 7.5]:
    ax.annotate('', xy=(cx, ys[1] + bh/2 + 0.04),
                xytext=(5.0, ys[0] - bh/2 - 0.04), arrowprops=kw, zorder=2)
for parent, children in [(2.5, [1.25, 3.75]), (7.5, [6.25, 8.75])]:
    for cx in children:
        ax.annotate('', xy=(cx, ys[2] + bh/2 + 0.04),
                    xytext=(parent, ys[1] - bh/2 - 0.04), arrowprops=kw, zorder=2)
child_xs = np.linspace(0.6, 9.4, 8)
for i, parent in enumerate([1.25, 3.75, 6.25, 8.75]):
    for cx in child_xs[i * 2:(i + 1) * 2]:
        ax.annotate('', xy=(cx, ys[3] + bh/2 + 0.04),
                    xytext=(parent, ys[2] - bh/2 - 0.04), arrowprops=kw, zorder=2)

# Right-side annotations
for y, lbl in zip(ys, ['层 0: 1×FFT(8)', '层 1: 2×FFT(4)  偶/奇拆分',
                        '层 2: 4×FFT(2)', '层 3: 8×FFT(1)  递归基']):
    ax.text(10.2, y, lbl, ha='left', va='center', fontsize=8.2, color='#2c3e50')

ax.set_xlim(-0.2, 13.5)
ax.set_ylim(-0.2, 4.2)
ax.axis('off')
ax.set_title('N=8  my_fft(x) 递归分治树\n'
             '向下拆分（偶/奇）→ N=1 基础情形 → 向上蝶形合并',
             fontsize=10.5, fontweight='bold', pad=4)
ax.legend(handles=[mpatches.Patch(fc=COLORS[n], ec='white', lw=1.5, alpha=0.88,
                                   label=f'FFT(N={n})') for n in [8, 4, 2, 1]],
          loc='lower right', fontsize=8.5, framealpha=0.9)
plt.tight_layout()
plt.show()

## 1. 递归结构：蝶形合并

Cooley-Tukey 核心恒等式：

```
X[k]     = E[k] + exp(-2πi·k/N) · O[k]      (k = 0..N/2-1)
X[k+N/2] = E[k] - exp(-2πi·k/N) · O[k]      (k = 0..N/2-1)
```

其中 `E = fft(x[::2])`（偶数下标），`O = fft(x[1::2])`（奇数下标）。
旋转因子 `exp(-2πi·k/N)` 乘完就是「蝶形（butterfly）」操作；`X[k+N/2]` 只是符号翻转，**不需要额外乘法**。
递归基：`N==1` 时 DFT 就是原值本身，直接返回。

In [ ]:
# 演示：手算 N=2 的 DFT vs FFT 公式
x2 = np.array([3.0, 7.0])
# DFT 定义: X[k] = sum_n x[n]*exp(-2pi*i*k*n/N)
X2_def = np.array([
    x2[0] + x2[1],           # k=0: exp(0)=1
    x2[0] - x2[1],           # k=1: exp(-pi*i)=-1
], dtype=complex)
X2_np = np.fft.fft(x2)
print('手算 X[0]:', X2_def[0], '  numpy:', X2_np[0])
print('手算 X[1]:', X2_def[1], '  numpy:', X2_np[1])
print('误差:', np.max(np.abs(X2_def - X2_np)))

## 2. 必须是 2 的幂；非 2 的幂用补零（zero padding）

Cooley-Tukey 每步把 N 对半分，要求 N 是 2 的幂次（`N = 2^k`）。
对任意长度 `L` 的信号，标准做法是找最小的 `N = 2^ceil(log2(L))`，在末尾补零到 N，变换后截取前 L 个系数——频率分辨率不变，只是增加了频率网格密度（细化）。

```
N_padded = 2 ** int(np.ceil(np.log2(len(x))))
x_padded = np.zeros(N_padded, dtype=complex)
x_padded[:len(x)] = x
```

In [ ]:
# 演示：N=5 补零到 8
x5 = np.array([1, 2, 3, 4, 5], dtype=float)
N_padded = 2 ** int(np.ceil(np.log2(len(x5))))
x_pad = np.zeros(N_padded)
x_pad[:len(x5)] = x5
print(f'原始长度: {len(x5)}  补零后: {N_padded}')
print('补零后:', x_pad)
print('np.fft.fft(补零后):', np.round(np.fft.fft(x_pad), 4))

## 3. 数值误差：浮点累积 < 1e-10

递归 FFT 有 `log₂N` 层，每层做浮点乘法；误差随层数线性累积，量级约 `N·log₂N·ε_machine`。
对 `N=1024`，double 精度下误差约 `1024·10·2.2e-16 ≈ 2e-12`，远低于 `1e-10`。
验证工具：`np.testing.assert_allclose(my_result, reference, atol=1e-10)`，失败时会打印最大偏差位置。

```
# atol 是绝对容忍，rtol 是相对容忍；对复数频谱两者都要设
np.testing.assert_allclose(A, B, rtol=1e-9, atol=1e-10)
```

In [ ]:
# 演示：量化不同 N 下的浮点误差（朴素 DFT vs numpy.fft，说明误差量级）
print("     N  max_abs_err (naive_dft vs numpy.fft)")
for N in [4, 16, 64, 256, 1024, 4096]:
    x = np.random.randn(N)
    ref = np.fft.fft(x)
    n_arr = np.arange(N)
    k_arr = n_arr.reshape((N, 1))
    M_naive = np.exp(-2j * np.pi * k_arr * n_arr / N)
    naive = M_naive @ x.astype(complex)
    err = np.max(np.abs(naive - ref))  # naive O(N²) 与 numpy 的差异
    print(f"{N:>6}  {err:.2e}")
print("\n(以上为朴素 DFT 与 numpy.fft 误差；实现 my_fft 后 cell 13 将真正验证其误差 < 1e-9)")


## 4. ✏️ 实现 `my_fft(x)`

**推理路线**：
1. 若 `len(x) == 1`，直接返回 `x.astype(complex)`（递归基）
2. 递归：`E = my_fft(x[::2])`，`O = my_fft(x[1::2])`
3. 构造旋转因子：`twiddle = exp(-2πi · np.arange(N//2) / N)`
4. 蝶形：`top = E + twiddle * O`；`bottom = E - twiddle * O`
5. 返回 `np.concatenate([top, bottom])`

**参考输入输出**：
```
x = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
my_fft(x)   # 应与下列结果误差 < 1e-10
np.fft.fft(x)
# [16.+0.j, -4.828-4.828j,  0.+0.j,  0.828-0.828j, 0.+0.j,  0.828+0.828j,  0.+0.j, -4.828+4.828j]
# (精确值: X[1]=X[7]*=-2√2(1+i)≈-4.828-4.828j; X[3]=X[5]*=√2(1-i)≈0.828-0.828j)
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def my_fft(x: np.ndarray) -> np.ndarray:
    """递归 Cooley-Tukey radix-2 FFT。x 长度必须是 2 的幂。"""
    N = len(x)
    # ✏️ TODO 1: 递归基 — N==1 时直接返回

    # ✏️ TODO 2: 递归拆分偶/奇下标
    E = ...  # my_fft(x[::2])
    O = ...  # my_fft(x[1::2])

    # ✏️ TODO 3: 旋转因子 twiddle = exp(-2pi*i * k / N), k = 0..N/2-1
    k = np.arange(N // 2)
    twiddle = ...

    # ✏️ TODO 4: 蝶形合并
    top    = ...  # E + twiddle * O
    bottom = ...  # E - twiddle * O

    # ✏️ TODO 5: 拼接返回
    return ...

In [ ]:
# ── 正确性检查 ──────────────────────────────────────────────
import numpy as np
x_demo = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
try:
    out_demo = my_fft(x_demo)
    if out_demo is None or out_demo is ...:
        print("⬜ 请先实现 my_fft 的各 TODO 项")
    else:
        ref_demo = np.fft.fft(x_demo)
        print('最大误差:', np.max(np.abs(out_demo - ref_demo)))
        all_pass = True
        for N in [4, 8, 16, 64, 256]:
            x = np.random.randn(N)
            if not np.allclose(my_fft(x), np.fft.fft(x), atol=1e-10):
                print(f'✗ N={N} 失败'); all_pass = False
        if all_pass:
            print('✅ 所有 N in [4,8,16,64,256] 通过，误差 < 1e-9')
except TypeError:
    print("⬜ 请先替换 ... 占位符后再运行检查格")
except RecursionError:
    print("⬜ my_fft 递归异常：检查递归基（N==1 时的 return）")


In [ ]:
# ── Aurora 连接：验证 my_fft 与 aurora_fft 数値等价 ─────────────────────────────
# aurora_fft 已在 cell 2 导入，此处做最终比较
try:
    x_check = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
    out_my = my_fft(x_check)
    if out_my is None or out_my is ...:
        print("□ 请先实现 my_fft 后再运行此格")
    else:
        out_aurora = aurora_fft(x_check)
        max_diff = float(np.max(np.abs(out_my - out_aurora)))
        np.testing.assert_allclose(out_my, out_aurora, atol=1e-10,
            err_msg="my_fft 与 aurora_fft 输出不一致，请检查蝶形合并逻辑")
        print(f"✅ my_fft ↔ aurora_fft 最大差异: {max_diff:.2e}  (< 1e-9)")
        print("   两者数値等价，算法路径不同（递归 vs 迭代 Cooley-Tukey）")
except TypeError:
    print("□ 请先替换 ... 占位符后再运行")
except RecursionError:
    print("□ my_fft 递归异常：检查递归基（N==1 时的 return）")


## 5. 参数实验：速度对比

- `N = 512`：naive DFT（O(N²)) vs `my_fft`（O(N log N)）vs `np.fft.fft`（C 扩展）
- 预期：`my_fft` 比 naive DFT 快约 `log₂(512) = 9` 倍；`np.fft.fft` 比 `my_fft` 快 10-50 倍（C 扩展 + SIMD）
- 调整 `N` 观察：倍数比例随 `N` 增大而增大（log 增益更显著）

In [ ]:
def naive_dft(x: np.ndarray) -> np.ndarray:
    """暴力 O(N²) DFT，用于速度基准对比。"""
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    M = np.exp(-2j * np.pi * k * n / N)
    return M @ x.astype(complex)

N = 512
x_bench = np.random.randn(N)

reps = 20
t0 = time.perf_counter()
for _ in range(reps): naive_dft(x_bench)
t_naive = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps): my_fft(x_bench)
t_my = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps): np.fft.fft(x_bench)
t_np = (time.perf_counter() - t0) / reps * 1000

print(f'N={N}  (平均 {reps} 次，单位 ms)')
print(f'  naive_dft : {t_naive:8.3f} ms')
print(f'  my_fft    : {t_my:8.3f} ms   ({t_naive/t_my:.1f}× faster than naive)')
print(f'  np.fft.fft: {t_np:8.3f} ms   ({t_my/t_np:.1f}× faster than my_fft)')
print(f'\n理论加速比 naive→FFT: log₂({N}) = {np.log2(N):.1f}')

---
→ **完成 `my_fft` 实现后，打开 [L42 · FFT 图形化](L42_visual_fft.ipynb)，对照蝴蝶图与各类信号的频谱形态，检验输出是否符合直觉，再回来。**

## 本课收束

`my_fft(x)` 通过递归蝶形操作输出 N 个复数频谱系数，误差控制在 `1e-9` 以内，与 `aurora.audio.transforms.fft` 在数值上等价，但算法路径独立：生产版采用**迭代式** Cooley-Tukey（位逆序置换 + 非递归蝴蝶），L41 将实现该迭代版本；两者都能正确驱动 STFT 的内层计算。下一课 L40 将在复数频谱上分析幅度谱与相位谱，验证频率分辨率与信号长度的关系。

---

→ **下一课**　[L40 · 频谱分析实战](L40_spectrum.ipynb)

> 下节课将学习 **频谱分析实战**：幅度谱 / 相位谱 / 频率分辨率，440Hz+880Hz 混合信号。